In [ ]:
# Databricks notebook source
# MAGIC
# MAGIC **What we build:**
# MAGIC - Synthetic customer master data linked to transactions
# MAGIC - SCD Type 2 pattern with effective_from, effective_to, is_current
# MAGIC - Simulate customer attribute changes over time
# MAGIC - Point-in-time queryable customer history
# MAGIC
# MAGIC **Why SCD Type 2?**
# MAGIC Regulators and fraud investigators need to know what a customer's
# MAGIC risk rating, address, or account status was at a specific point in time.
# MAGIC SCD Type 1 (overwrite) destroys that history. SCD Type 2 preserves it.

In [ ]:
SP_CLIENT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-id")
SP_TENANT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-tenant-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-secret")

STORAGE_ACCOUNT  = "retailbankingdl"
ADLS_SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"
ADLS_BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"

SILVER_TRANSACTIONS = f"{ADLS_SILVER_PATH}/transactions_cleaned"
SILVER_CUSTOMERS    = f"{ADLS_SILVER_PATH}/customers_scd2"

print("✅ Configuration loaded")

In [ ]:
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_ID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_SECRET
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
)

print("✅ ADLS Gen2 connection configured")

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta
from pyspark.sql.functions import col, lit, when, to_date
from pyspark.sql.types import *

random.seed(42)

# Parameters
NUM_CUSTOMERS = 1000
RISK_RATINGS  = ["LOW", "MEDIUM", "HIGH"]
COUNTRIES     = ["US", "UK", "DE", "FR", "CH", "SG", "JP"]
ACCOUNT_TYPES = ["PERSONAL", "BUSINESS", "PREMIUM"]
STATUSES      = ["ACTIVE", "SUSPENDED", "CLOSED"]

def random_date(start_year=2019, end_year=2023):
    start = datetime(start_year, 1, 1)
    end   = datetime(end_year, 12, 31)
    return start + timedelta(days=random.randint(0, (end - start).days))

# Generate base customer records (Version 1 — original state)
customers_v1 = []
for i in range(1, NUM_CUSTOMERS + 1):
    open_date = random_date(2019, 2021)
    customers_v1.append({
        "customer_id":      f"C{str(i).zfill(5)}",
        "account_type":     random.choice(ACCOUNT_TYPES),
        "country":          random.choice(COUNTRIES),
        "risk_rating":      random.choice(RISK_RATINGS),
        "account_status":   "ACTIVE",
        "credit_limit":     round(random.uniform(1000, 50000), 2),
        "effective_from":   open_date.strftime("%Y-%m-%d"),
        "effective_to":     "9999-12-31",
        "is_current":       True,
        "scd_version":      1
    })

print(f"✅ Generated {len(customers_v1):,} base customer records (Version 1)")

In [ ]:
# Simulate changes for 30% of customers
changed_customers = random.sample(customers_v1, int(NUM_CUSTOMERS * 0.30))

customers_v2 = []
for c in changed_customers:
    change_date = random_date(2022, 2023)

    # Close Version 1 record
    c["effective_to"] = change_date.strftime("%Y-%m-%d")
    c["is_current"]   = False

    # Create Version 2 record with changed attributes
    new_risk = random.choice([r for r in RISK_RATINGS if r != c["risk_rating"]])
    new_status = random.choice(STATUSES)

    customers_v2.append({
        "customer_id":    c["customer_id"],
        "account_type":   c["account_type"],
        "country":        c["country"],
        "risk_rating":    new_risk,
        "account_status": new_status,
        "credit_limit":   round(c["credit_limit"] * random.uniform(0.8, 1.5), 2),
        "effective_from": change_date.strftime("%Y-%m-%d"),
        "effective_to":   "9999-12-31",
        "is_current":     True,
        "scd_version":    2
    })

print(f"✅ Simulated changes for {len(customers_v2):,} customers (Version 2 records)")
print(f"   Total SCD records: {len(customers_v1) + len(customers_v2):,}")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType, IntegerType
from pyspark.sql.functions import current_timestamp, sha2, concat_ws
from datetime import datetime

# Combine V1 and V2 records
all_customers = customers_v1 + customers_v2

schema = StructType([
    StructField("customer_id",    StringType(),  False),
    StructField("account_type",   StringType(),  True),
    StructField("country",        StringType(),  True),
    StructField("risk_rating",    StringType(),  True),
    StructField("account_status", StringType(),  True),
    StructField("credit_limit",   DoubleType(),  True),
    StructField("effective_from", StringType(),  False),
    StructField("effective_to",   StringType(),  False),
    StructField("is_current",     BooleanType(), False),
    StructField("scd_version",    IntegerType(), False),
])

df_customers = spark.createDataFrame(all_customers, schema)

# Add surrogate key (unique per SCD row)
df_customers = df_customers.withColumn(
    "customer_sk",
    sha2(concat_ws("||",
        col("customer_id"),
        col("effective_from"),
        col("scd_version").cast("string")
    ), 256)
)

# Add audit columns
ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")
df_customers = (df_customers
    .withColumn("silver_ingestion_ts",   current_timestamp())
    .withColumn("silver_ingestion_date", lit(ingestion_date))
    .withColumn("silver_source",         lit("synthetic_customer_generator"))
)

total = df_customers.count()
current_records = df_customers.filter(col("is_current") == True).count()
historical_records = df_customers.filter(col("is_current") == False).count()

print(f"✅ SCD Type 2 DataFrame built")
print(f"   Total records:      {total:,}")
print(f"   Current records:    {current_records:,}  (is_current = True)")
print(f"   Historical records: {historical_records:,}  (is_current = False)")

In [ ]:
print("Sample — customer with history (2 versions):\n")

# Find a customer with 2 versions
multi_version = df_customers.groupBy("customer_id").count().filter(col("count") > 1).limit(1)
sample_id = multi_version.collect()[0]["customer_id"]

df_customers.filter(col("customer_id") == sample_id).select(
    "customer_id", "risk_rating", "account_status",
    "effective_from", "effective_to", "is_current", "scd_version"
).orderBy("scd_version").show(truncate=False)

print("\nThis shows the full history for one customer.")
print("Version 1: original record (is_current=False, effective_to=change date)")
print("Version 2: updated record  (is_current=True,  effective_to=9999-12-31)")

In [ ]:
print(f"Writing to Silver Delta table...")
print(f"  Target: {SILVER_CUSTOMERS}")

(df_customers
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("silver_ingestion_date")
    .save(SILVER_CUSTOMERS)
)

print("✅ Silver customer SCD2 table written")

In [ ]:
print("Demonstrating point-in-time query capability...\n")

df_verify = spark.read.format("delta").load(SILVER_CUSTOMERS)

# Query: what was the risk rating of each customer on 2022-06-01?
query_date = "2022-06-01"

df_point_in_time = df_verify.filter(
    (col("effective_from") <= query_date) &
    (col("effective_to")   >  query_date)
)

pit_count = df_point_in_time.count()
print(f"  ✅ Point-in-time query for {query_date}: {pit_count:,} customers found")
print(f"\nRisk rating distribution on {query_date}:")
df_point_in_time.groupBy("risk_rating").count().orderBy("risk_rating").show()

In [ ]:
print("Current customer snapshot (is_current = True):\n")

df_current = df_verify.filter(col("is_current") == True)
print(f"  ✅ Current customers: {df_current.count():,}")
print(f"\nRisk rating distribution (current):")
df_current.groupBy("risk_rating").count().orderBy("risk_rating").show()
print(f"\nAccount status distribution (current):")
df_current.groupBy("account_status").count().orderBy("account_status").show()

In [ ]:
total        = df_verify.count()
current      = df_verify.filter(col("is_current") == True).count()
historical   = df_verify.filter(col("is_current") == False).count()

print("=" * 60)
print("SILVER LAYER — SCD TYPE 2 CUSTOMER DIMENSION COMPLETE")
print("=" * 60)
print(f"  ✅ Total SCD records:      {total:,}")
print(f"  ✅ Current records:        {current:,}")
print(f"  ✅ Historical records:     {historical:,}")
print(f"  ✅ Point-in-time queries:  supported")
print(f"  ✅ Location: {SILVER_CUSTOMERS}")
print("=" * 60)
print("SCD Type 2 pattern implemented.")
print("Ready for Week 2 Day 4 — Data Quality Checks")
print("=" * 60)